In [11]:
import keras
import datasets
import numpy as np
import transformers
import sklearn.metrics
import tensorflow as tf
import tqdm.notebook as tqdm
import sklearn.model_selection
import matplotlib.pyplot as plt

In [12]:
pip install -U jupyter ipywidgets notebook

Note: you may need to restart the kernel to use updated packages.


In [13]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except:
        pass

In [14]:
dataset = datasets.load_dataset(
    'google-research-datasets/go_emotions', name='raw', split='train'
)

emotions = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]

In [15]:
# 2. Токенайзер
# ========================
tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch_texts, max_len=64):
    return tokenizer(
        batch_texts,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="np"
    )["input_ids"]

# ========================
# 3. Train/Test Split
# ========================
texts = dataset["text"]
labels = np.array(dataset["labels"])  # мульти-лейблы

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

X_train_tok = tokenize(X_train)
X_test_tok = tokenize(X_test)

train_dataset = tf.data.Dataset.from_tensor_slices((X_train_tok, y_train)).batch(32)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test_tok, y_test)).batch(32)

ValueError: Column 'labels' doesn't exist.

In [6]:
def get_name(prefix: str | None = None, suffix: str | None = None, separator: str = '_') -> str | None:
    return prefix and prefix + separator + suffix or suffix or None

In [7]:
def get_model(
    units: int,
    n_tokens: int,
    n_labels: int,
    n_stacks: int = 1,
    bidirectional: bool = False,
    name: str | None = None,
    cell_type: type[keras.layers.Layer] = keras.layers.LSTMCell
) -> keras.Model:
    """
    Creates a model with RNN architecture for sequence multilabel classification.
    """
    inputs = keras.Input(shape=(None,), dtype="int32", name="tokens")
    x = keras.layers.Embedding(input_dim=n_tokens, output_dim=units, mask_zero=True)(inputs)

    # RNN stack
    for i in range(n_stacks):
        if cell_type == keras.layers.LSTMCell:
            rnn_layer = keras.layers.LSTM(units, return_sequences=(i < n_stacks - 1))
        else:
            rnn_layer = keras.layers.GRU(units, return_sequences=(i < n_stacks - 1))

        if bidirectional:
            x = keras.layers.Bidirectional(rnn_layer)(x)
        else:
            x = rnn_layer(x)

    outputs = keras.layers.Dense(n_labels, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name=name or "rnn_model")

In [8]:
models = [
    get_model(
        units=128,
        n_tokens=len(tokenizer.get_vocab()),
        n_labels=len(emotions),
        name="lstm_single",
        bidirectional=False,
        n_stacks=1,
        cell_type=keras.layers.LSTMCell
    ),
    get_model(
        units=128,
        n_tokens=len(tokenizer.get_vocab()),
        n_labels=len(emotions),
        name="bigru_stack",
        bidirectional=True,
        n_stacks=2,
        cell_type=keras.layers.GRUCell
    ),
]


In [9]:
for model in models:
    model.compile(
        loss=keras.losses.BinaryCrossentropy(from_logits=False),
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        metrics=[
            keras.metrics.BinaryAccuracy(name="acc"),
            keras.metrics.AUC(name="auc", multi_label=True)
        ]
    )


In [10]:
for train_dataset, test_dataset in datasets:
    for model in models:
        model.fit(train_dataset, validation_data=test_dataset, epochs=...)

TypeError: 'module' object is not iterable

In [ ]:
from sklearn.metrics import roc_curve, auc, roc_auc_score

def plot_roc_curve(
    X: np.ndarray,
    y: np.ndarray,
    model: keras.Model,
    ax: plt.Axes | None = None
) -> float:
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    y_pred = model.predict(X)
    aucs = []
    for i in range(y.shape[1]):
        fpr, tpr, _ = roc_curve(y[:, i], y_pred[:, i])
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        ax.plot(fpr, tpr, lw=1, alpha=0.7, label=f"{emotions[i]} (AUC={roc_auc:.2f})")

    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC curves")
    ax.legend(fontsize=8, ncol=2)
    return float(np.mean(aucs))


In [ ]:
def label_text(text: str, model: keras.Model, threshold: float = 0.5, max_length: int | None = None) -> list[str]:
    tokens = tokenizer(
        text,
        return_tensors="np",
        padding="max_length" if max_length else True,
        truncation=True,
        max_length=max_length
    )["input_ids"]

    probs = model.predict(tokens)[0]
    return [emo for emo, p in zip(emotions, probs) if p >= threshold]


In [ ]:
def plot_emotion_scores(text: str, model: keras.Model, max_length: int | None = None, ax: plt.Axes | None = None):
    tokens = tokenizer(
        text,
        return_tensors="np",
        padding="max_length" if max_length else True,
        truncation=True,
        max_length=max_length
    )["input_ids"]

    probs = model.predict(tokens)[0]
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    ax.barh(emotions, probs)
    ax.set_title(f"Emotion probabilities for: {text[:50]}...")
    ax.set_xlabel("Probability")
    ax.set_xlim(0, 1)
    plt.tight_layout()
